In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="jazza234234/david-dataset", 
    repo_type="dataset", local_dir="./david-dataset", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]


'/home/ubuntu/david-dataset'

In [4]:
files = glob('david-dataset/*/*.parquet')
len(files)

1

In [ ]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [7]:
# data = loop((files[:1], 0))
# data

In [8]:
with open('david.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('david-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq david-dataset_audio.zip david-dataset_audio
# !hf upload malaysia-ai/Multilingual-TTS david-dataset_audio.zip --repo-type=dataset

In [16]:
# !zip -rq david-dataset_audio_neucodec.zip david-dataset_audio_neucodec
# !hf upload malaysia-ai/Multilingual-TTS david-dataset_audio_neucodec.zip --repo-type=dataset

In [12]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'david-dataset_audio/david-dataset-data-train-00000-of-00001_0.mp3',
 'text': 'my witness statement.',
 'speaker': 'david-dataset_audio'}

In [13]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'david-dataset')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 853.54ba/s]
Processing Files (1 / 1): 100%|██████████| 65.1kB / 65.1kB,  0.00B/s  
New Data Upload: 100%|██████████| 65.1kB / 65.1kB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  2.56 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/8d9f98f69fb0d3d75c98edbd9b330c53fadd16de', commit_message='Upload dataset', commit_description='', oid='8d9f98f69fb0d3d75c98edbd9b330c53fadd16de', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)